### Muntatge de Google Drive a Colab
Aquest codi connecta Google Drive amb Google Colab, fent que els fitxers del Drive siguin accessibles des del camí `/content/drive1`.




In [2]:
from google.colab import drive
drive.mount('/content/drive1')

Mounted at /content/drive1


In [1]:
import os
import time
import torch
import numpy as np
import pandas as pd
import torchvision
import matplotlib.pyplot as plt

from tqdm import tqdm
from matplotlib import gridspec
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

### Configuració de rutes i paràmetres d’avaluació
Defineix on es troben els **models** (`MODELS_DIR`) i on es guardaran els **resultats** (`OUTPUT_DIR`).  

Configura el **dispositiu** (`GPU/CPU`), el **llindar mínim de score** per acceptar prediccions (`SCORE_THRESH`) i els **nivells de falsos positius per imatge** (`FP_LEVELS`) per calcular la mètrica tipus NODE21.  

Inclou també un criteri d’**early stop** (`EARLY_STOP`) per descartar ràpidament models dolents si, després d’un mínim d’imatges, tenen **baixa sensibilitat**, **massa falsos positius** o **IoU mitjana molt baixa**.


In [5]:
MODELS_DIR = "/content/drive1/MyDrive/ML/DATASETS/NODE21/MODELS1"
OUTPUT_DIR = "/content/drive1/MyDrive/ML/DATASETS/NODE21/RESULTS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SCORE_THRESH = 0.01

FP_LEVELS = (0.25, 0.5, 1, 2, 4, 8)

EARLY_STOP = {
    "min_images": 40,
    "min_sensitivity": 0.15,
    "max_fp_per_image": 20.0,
    "min_mean_iou": 0.05
}

### Definició de rutes del dataset NODE21
Defineix la ruta base del dataset processat (`BASE_PATH`) i construeix les rutes cap a:

- la carpeta d’imatges en format **PNG** (`PATH_IMAGES`)
- el fitxer de metadades **CSV** (`PATH_METADATA`), que conté la informació de les anotacions (caixes, labels, etc.)

Finalment, imprimeix aquestes rutes per verificar que apunten al lloc correcte.


In [6]:
BASE_PATH = "/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data"

PATH_IMAGES = f"{BASE_PATH}/images"
PATH_METADATA = f"{BASE_PATH}/metadata_augmented.csv"
PATH_METADATA = "/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata_augmented_def2.csv"
#PATH_METADATA = "/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata.csv"

print(PATH_IMAGES)
print(PATH_METADATA)

/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/images
/content/drive1/MyDrive/ML/DATASETS/NODE21/proccessed_data/metadata_augmented_def2.csv


### Dataset de detecció NODE21 (PNG)
Defineix un `Dataset` de PyTorch que:

- Filtra el `DataFrame` per quedar-se només amb imatges **existents** al disc.
- Manté una llista d’índexs reals (`indices`) per accedir al `DataFrame` sense perdre l’ordre.
- Carrega cada imatge en **grisos**, la normalitza a **[0,1]** i la converteix a **3 canals**.
- Recupera totes les anotacions (boxes) associades a aquella imatge i construeix el diccionari `target` per a models tipus Faster R-CNN / RetinaNet.


In [7]:
import os
import torch
from torch.utils.data import Dataset
import numpy as np
import cv2

class Node21DetectionDatasetPNG(Dataset):
    def __init__(self, df):
        # Hacemos una copia para no modificar el df original
        self.df = df.copy()

        # Filtro: solo filas con la ruta válida
        self.df = self.df[self.df["file_path"].apply(os.path.exists)]

        # Lista de indices del df, en su orden natural (MUY IMPORTANTE)
        self.indices = self.df.index.tolist()


    def __len__(self):
        return len(self.indices)


    def __getitem__(self, idx):
        # Obtener el índice REAL en el DataFrame
        real_idx = self.indices[idx]

        # Extraer toda la fila para esa imagen
        row = self.df.loc[real_idx]
        path = row["file_path"]

        # ===============================
        # 1. Cargar imagen PNG
        # ===============================
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = img.astype(np.float32)

        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

        # Convertir a 3 canales
        img3 = np.stack([img, img, img], axis=0)

        # ===============================
        # 2. Obtener TODAS las anotaciones de esta imagen
        # ===============================
        rows = self.df[self.df["file_path"] == path]

        boxes = []
        labels = []

        for _, r in rows.iterrows():
            if r["label"] == 1:
                x1 = float(r["x"])
                y1 = float(r["y"])
                x2 = x1 + float(r["width"])
                y2 = y1 + float(r["height"])
                boxes.append([x1, y1, x2, y2])
                labels.append(1)

        if len(boxes) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
            iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "area": area,
            "iscrowd": iscrowd,
            "image_id": torch.tensor([real_idx])
        }

        return torch.tensor(img3, dtype=torch.float32), target

### Càrrega del metadata i recompte de classes
Aquest codi:

- Llegeix el fitxer CSV de metadades (`PATH_METADATA`) en un `DataFrame`.
- Construeix la ruta real de cada imatge PNG a la columna `file_path`.
- Compta quantes mostres són **negatives** (`label=0`) i quantes són **positives** (`label=1`) per veure el **desbalanceig** del dataset.


In [8]:
import pandas as pd
import os

df = pd.read_csv(PATH_METADATA)

df["file_path"] = df["img_name"].apply(
    lambda x: f"{PATH_IMAGES}/{x.replace('.mha', '.png')}"
)

df.head()
num_neg = (df["label"] == 0).sum()
num_pos = (df["label"] == 1).sum()
print("Negativos:", num_neg)
print("Positivos:", num_pos)

Negativos: 3748
Positivos: 4427


### Preparació de Dataloaders (train/val) per detecció
Aquest bloc:

- Defineix un `collate_fn` per poder agrupar imatges i targets (llistes) en detecció.
- Divideix el `DataFrame` en **train (80%)** i **validació (20%)**.
- Crea els `Dataset` de NODE21 per a cada partició.
- Genera els `DataLoader` per entrenar i validar amb `batch_size=2`.


In [9]:
# 1. Collate
def detection_collate(batch):
    imgs = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    return imgs, targets

# 2. Split
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 3. Datasets
train_dataset = Node21DetectionDatasetPNG(train_df)
val_dataset   = Node21DetectionDatasetPNG(val_df)

# 4. Dataloaders
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True,
                          collate_fn=detection_collate,
                          num_workers=2, pin_memory=True)

val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False,
                          collate_fn=detection_collate,
                          num_workers=2, pin_memory=True)

### Selecció automàtica d’arquitectura segons el nom del checkpoint
Aquesta funció detecta quin tipus de model s’ha de construir mirant el **nom del fitxer** del checkpoint:

- Si el nom conté `"efficientnet"` → retorna **`frcnn_efficientnet`**
- Si no → assumeix que és un **Faster R-CNN amb ResNet** i retorna **`frcnn_resnet`**

In [10]:
def get_model_builder(checkpoint_name):
    if "efficientnet" in checkpoint_name.lower():
        return "frcnn_efficientnet"
    else:
        return "frcnn_resnet"

### Creació del backbone EfficientNet-B0 (sense pesos preentrenats)
Aquesta funció carrega una **EfficientNet-B0** i en recupera només la part de *features* per usar-la com a **backbone** en detecció.  
A més, fixa `out_channels = 1280` perquè Faster R-CNN sàpiga quants canals de sortida genera el backbone.


In [11]:
from torchvision.models import efficientnet_b0

def build_my_efficientnet_backbone():
    model = efficientnet_b0(weights=None)
    backbone = model.features
    backbone.out_channels = 1280
    return backbone

### Faster R-CNN amb backbone EfficientNet (adaptat a 2 classes)
Aquesta funció construeix un **Faster R-CNN** utilitzant un backbone **EfficientNet**.  
Defineix un **AnchorGenerator personalitzat** (una escala i 3 aspect ratios) per a la RPN, i configura el model per detectar **2 classes** (fons + nòdul).


In [12]:
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator

def build_frcnn_efficientnet(backbone):
    backbone.out_channels = 1280

    anchor_generator = AnchorGenerator(
        sizes=((128,),),              # 1 escala
        aspect_ratios=((0.5, 1.0, 2.0),)  # 3 ratios
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=2,
        rpn_anchor_generator=anchor_generator
    )
    return model


### Faster R-CNN amb backbone ResNet50 + FPN (2 classes)
Aquesta funció crea un **Faster R-CNN ResNet50-FPN** sense pesos preentrenats.  
Després substitueix el `box_predictor` per adaptar la sortida a **2 classes** (fons + nòdul).


In [13]:
def build_frcnn_resnet():
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    return model

### Construcció d’un Faster R-CNN ResNet50-FPN (classes configurables)
Aquesta funció crea un model **Faster R-CNN amb ResNet50 + FPN** sense pesos preentrenats.  
Després substitueix el `box_predictor` per ajustar-lo al nombre de classes (`num_classes`), per exemple **2** (fons + nòdul).


In [14]:
def build_model(num_classes=2):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features, num_classes
    )

    return model

### Construcció del model per inferència local (2 classes)
Aquesta funció crea un **Faster R-CNN ResNet50-FPN** sense pesos preentrenats i li posa un `box_predictor` per **2 classes** (fons + nòdul).  
El model es mou al `device` i es deixa en mode **eval()**, pensat per fer **inferència/càrrega de checkpoints** i no entrenament.


In [15]:
def build_model_for_local_focal(device):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None
    )

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(
        in_features, 2
    )

    model.to(device)
    model.eval()
    return model


### Càrrega segura d’un checkpoint (compatible o descartat)
Aquesta funció intenta carregar un **checkpoint** dins del model amb `strict=False`.  
Si falten pesos o en sobren (model **incompatible**) o hi ha un error, mostra el motiu i retorna **False**.  
Si tot encaixa correctament, retorna **True**.


In [16]:
def load_checkpoint_safe(model, path, device):
    try:
        state = torch.load(path, map_location=device)
        missing, unexpected = model.load_state_dict(state, strict=False)

        if missing or unexpected:
            print("   Incompatible")
            return False

        return True
    except Exception as e:
        print("   Error:", e)
        return False

### Càlcul de la IoU (Intersection over Union)
Aquesta funció calcula la **IoU** entre dues bounding boxes.  
Primer calcula l’àrea de la intersecció i després la divideix per l’àrea de la unió.  
Retorna un valor entre **0 i 1** (0 = no se solapen, 1 = solapament perfecte).


In [17]:
def compute_iou(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])

    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (a[2] - a[0]) * (a[3] - a[1])
    areaB = (b[2] - b[0]) * (b[3] - b[1])

    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

### Recollida de deteccions amb *early stop*
Aquesta funció recorre tot el *dataset* i fa inferència per obtenir **prediccions (boxes + scores)** i **GT** per cada imatge.  
Calcula **TP/FP** amb un *matching* per IoU i acumula una IoU mitjana.  
Si després d’un mínim d’imatges el model té **sensibilitat baixa**, **massa FP/imatge** o **IoU mitjana baixa**, s’atura i retorna **None** per descartar el checkpoint ràpidament.


In [18]:
def collect_detections_early_stop(
    model, dataset, device, score_thresh, early_cfg
):
    model.eval()

    detections = []
    gt_count = tp = fp = iou_sum = iou_n = 0
    n_images = len(dataset)

    with torch.no_grad():
        for i in range(n_images):
            img, target = dataset[i]

            # IMAGEN A GPU
            img = img.to(device, non_blocking=True)

            gt_boxes = target["boxes"].cpu().numpy()
            gt_count += len(gt_boxes)

            # INFERENCIA EN CUDA
            pred = model([img])[0]

            boxes = pred["boxes"].detach().cpu().numpy()
            scores = pred["scores"].detach().cpu().numpy()

            keep = scores >= score_thresh
            boxes = boxes[keep]
            scores = scores[keep]

            # matching CPU
            matched = set()
            for pb in boxes:
                best_iou, best_gt = 0.0, -1
                for j, gt in enumerate(gt_boxes):
                    if j in matched:
                        continue
                    iou = compute_iou(pb, gt)
                    if iou > best_iou:
                        best_iou, best_gt = iou, j

                if best_iou >= 0.2:
                    tp += 1
                    matched.add(best_gt)
                    iou_sum += best_iou
                    iou_n += 1
                else:
                    fp += 1

            detections.append({
                "gt_boxes": gt_boxes,
                "pred_boxes": boxes,
                "scores": scores
            })


            if i % 50 == 0 or i == n_images - 1:
                print(
                    f"    Inferencia: {i+1}/{n_images} imágenes",
                    end="\r"
                )

            # -------- EARLY STOP --------
            if i + 1 >= early_cfg["min_images"]:
                sens = tp / (gt_count + 1e-8)
                fp_img = fp / (i + 1)
                mean_iou = iou_sum / iou_n if iou_n else 0.0

                if (
                    sens < early_cfg["min_sensitivity"]
                    or fp_img > early_cfg["max_fp_per_image"]
                    or mean_iou < early_cfg["min_mean_iou"]
                ):
                    print(
                        f"\nEarly stop @ {i+1} | "
                        f"Sens={sens:.3f} | "
                        f"FP/img={fp_img:.1f} | "
                        f"IoU={mean_iou:.3f}"
                    )
                    return None, None

    print()
    return detections, gt_count




### Càlcul de la corba FROC (FP/imatge vs sensibilitat)
Aquesta funció construeix la **corba FROC** a partir de les deteccions del model.  
Ordena les prediccions per **score**, fa *matching* amb les caixes GT amb un llindar d’**IoU**, i marca cada predicció com a **TP** o **FP**.  
Finalment calcula la **sensibilitat acumulada** i els **falsos positius per imatge**, retornant `fps` i `sens` per poder dibuixar la corba.


In [19]:
def compute_froc(detections, iou_thresh=0.2):
    all_tp, all_fp, all_scores = [], [], []
    total_gt = sum(len(d["gt_boxes"]) for d in detections)
    num_images = len(detections)

    for det in detections:
        matched = set()
        boxes, scores = det["pred_boxes"], det["scores"]

        order = np.argsort(scores)[::-1]
        boxes, scores = boxes[order], scores[order]

        for i, pb in enumerate(boxes):
            best_iou, best_gt = 0.0, -1
            for j, gt in enumerate(det["gt_boxes"]):
                if j in matched:
                    continue
                iou = compute_iou(pb, gt)
                if iou > best_iou:
                    best_iou, best_gt = iou, j

            if best_iou >= iou_thresh:
                all_tp.append(1)
                all_fp.append(0)
                matched.add(best_gt)
            else:
                all_tp.append(0)
                all_fp.append(1)

            all_scores.append(scores[i])

    order = np.argsort(all_scores)[::-1]
    tp = np.cumsum(np.array(all_tp)[order])
    fp = np.cumsum(np.array(all_fp)[order])

    sens = tp / (total_gt + 1e-8)
    fps = fp / num_images

    return fps, sens


### Avaluació completa del model amb NODE21 score (FROC)
Aquesta funció avalua el model sobre el *dataset* calculant la corba **FROC** i les sensibilitats a diferents nivells de **FP/imatge**.  
Si el model és molt dolent, pot activar un **early-stop** per evitar perdre temps.  
Finalment calcula el **NODE21 score** (mitjana de sensibilitats als FP definits) i genera una gràfica FROC amb els punts destacats.


In [20]:
def evaluate_model(model, dataset, device, name):
    model.eval()
    detections, gt_count = collect_detections_early_stop(
        model, dataset, device, SCORE_THRESH, EARLY_STOP
    )

    if detections is None:
        return {"early_stopped": True}

    fps, sens = compute_froc(detections)

    sens_at_fp = {
        fp: np.max(sens[fps <= fp]) if np.any(fps <= fp) else 0.0
        for fp in FP_LEVELS
    }

    node21 = float(np.mean(list(sens_at_fp.values())))

    # -------- Plot --------
    fig = plt.figure(figsize=(8, 6))
    plt.plot(fps, sens, lw=2)
    for fp, s in sens_at_fp.items():
        plt.scatter(fp, s)
        plt.text(fp, s, f"{s:.2f}", fontsize=9)

    plt.xscale("log")
    plt.xlabel("FP / image")
    plt.ylabel("Sensitivity")
    plt.title(name)
    plt.grid(True, which="both")
    plt.tight_layout()

    return {
        "node21_score": node21,
        "sensitivities": sens_at_fp,
        "figure": fig
    }


### Inferència automàtica del backbone a partir d’un checkpoint
Aquesta funció llegeix el *state_dict* del checkpoint i mira la forma dels pesos de la capa **RPN conv**.  
Si detecta **1280 canals**, assumeix que el model és **Faster R-CNN + EfficientNet**; si detecta **256 canals**, assumeix **Faster R-CNN + ResNet50-FPN**.  
Si no troba cap patró compatible, llença un error perquè no pot deduir l’arquitectura.


In [22]:
def infer_builder_from_checkpoint(path, device):
    state = torch.load(path, map_location=device)
    # RPN conv: rpn.head.conv.0.0.weight
    for k, v in state.items():
        if "rpn.head.conv" in k and "weight" in k:
            in_ch = v.shape[0]
            if in_ch == 1280:
                return "frcnn_efficientnet"
            elif in_ch == 256:
                return "frcnn_resnet"
    raise ValueError("No se puede inferir el backbone del checkpoint")


### Bloc CBAM (Channel + Spatial Attention)
Aquest mòdul implementa **CBAM**, que aplica atenció en dos passos:

- **Channel Attention**: calcula quins **canals** són més importants (mitjançant *avg pooling* i *max pooling* globals + un MLP) i reescala cada canal.
- **Spatial Attention**: calcula quines **zones de la imatge** són més rellevants (a partir d’avg i max per píxel + una convolució 7×7) i reescala l’activació espacialment.

El resultat és un mapa de característiques que destaca **els canals útils** i **les regions útils** per a la detecció.


In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )

        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)

    def forward(self, x):
        # Channel attention
        avg = torch.mean(x, dim=(2, 3))
        max_ = torch.amax(x, dim=(2, 3))
        ca = torch.sigmoid(self.mlp(avg) + self.mlp(max_))
        x = x * ca.unsqueeze(-1).unsqueeze(-1)

        # Spatial attention
        avg = torch.mean(x, dim=1, keepdim=True)
        max_, _ = torch.max(x, dim=1, keepdim=True)
        sa = torch.sigmoid(self.conv_spatial(torch.cat([avg, max_], dim=1)))

        return x * sa


### Backbone ResNet amb CBAM a l’últim nivell (C5)
Aquesta classe crea un *backbone* basat en **ResNet**, però afegeix un mòdul **CBAM** només després de `layer4` (features **C5**, 2048 canals).

Durant el `forward`, es calcula tota la piràmide de característiques:
- `out["0"]` = sortida de **layer1** (C2)
- `out["1"]` = sortida de **layer2** (C3)
- `out["2"]` = sortida de **layer3** (C4)
- `out["3"]` = sortida de **layer4** amb **CBAM aplicat** (C5)

Això permet que la FPN/Detector rebi multiescala, però amb **atenció extra** només al nivell més profund i semàntic.


In [61]:
import torch.nn as nn
from collections import OrderedDict

class ResNetBackboneWithCBAM(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.body = backbone
        self.cbam = CBAM(channels=2048)

    def forward(self, x):
        out = OrderedDict()

        x = self.body.conv1(x)
        x = self.body.bn1(x)
        x = self.body.relu(x)
        x = self.body.maxpool(x)

        x = self.body.layer1(x)
        out["0"] = x          # C2

        x = self.body.layer2(x)
        out["1"] = x          # C3

        x = self.body.layer3(x)
        out["2"] = x          # C4

        x = self.body.layer4(x)
        x = self.cbam(x)      # CBAM SOLO AQUÍ
        out["3"] = x          # C5

        return out



### Backbone amb CBAM + Feature Pyramid Network (FPN)
Aquesta classe defineix un *backbone* complet per detecció, combinant:

- **ResNet amb CBAM** (aplicat a `layer4`, és a dir, C5)
- **FPN** per generar representacions multiescala consistents per al detector

La FPN rep les sortides del backbone en 4 nivells:
- C2 (256 canals)
- C3 (512 canals)
- C4 (1024 canals)
- C5 (2048 canals, amb CBAM)

I les transforma totes a **256 canals**, generant mapes multiresolució preparats per Faster R-CNN o RetinaNet.

Finalment, `out_channels = 256` indica que totes les sortides de la piràmide tenen aquest nombre de canals.


In [56]:
from torchvision.ops import FeaturePyramidNetwork

class CBAMBackboneWithFPN(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = ResNetBackboneWithCBAM()

        self.fpn = FeaturePyramidNetwork(
            in_channels_list=[256, 512, 1024, 2048],
            out_channels=256
        )

        self.out_channels = 256

    def forward(self, x):
        features = self.body(x)
        return self.fpn(features)


### Construcció d’un Faster R-CNN amb ResNet50+FPN i CBAM al backbone
Aquesta funció crea un model **Faster R-CNN ResNet50-FPN** i modifica el *backbone* per afegir-hi un mòdul **CBAM** (atenció) a la part final de la ResNet.

Concretament:
- Carrega un **Faster R-CNN ResNet50-FPN** sense pesos preentrenats.
- Substitueix `model.backbone.body` per una versió personalitzada (**ResNetBackboneWithCBAM**) que aplica CBAM a `layer4` (C5).
- Canvia el *predictor* final perquè el model tingui **2 classes** (fons + nòdul).
- Envia el model a GPU/CPU i el deixa en mode **avaluació (`eval`)**.


In [62]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def build_frcnn_resnet_cbam(device):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None
    )

    # Insertar CBAM en el backbone
    old_body = model.backbone.body
    model.backbone.body = ResNetBackboneWithCBAM(old_body)

    # Head Node21
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)

    model.to(device)
    model.eval()
    return model




### Avaluació massiva de checkpoints **CBAM** (comparació NODE21)
Aquest codi recorre tots els *checkpoints* `.pth` que contenen `"cbam"` al nom i en fa una **avaluació automàtica** sobre el conjunt de validació.

Per a cada model:
- Construeix el **Faster R-CNN amb CBAM** (mateixa arquitectura que a l’entrenament).
- Carrega el *checkpoint* i comprova que realment conté **pesos del CBAM**.
- Descarta models amb moltes incompatibilitats (`missing/unexpected`) per evitar avaluacions incorrectes.
- Avalua el model amb la mètrica **NODE21 (FROC sensitivities)** i pot fer *early stop* si el model és degenerat.
- Guarda la figura **FROC** a disc i registra els resultats (score, temps, sensibilitats).


In [ ]:
results = []
skipped = []

model_files = [
    f for f in os.listdir(MODELS_DIR)
    if f.endswith(".pth") and "cbam" in f.lower()
]

start_all = time.time()

for fname in tqdm(model_files, desc="Evaluando modelos CBAM"):
    print(f"\n {fname}")

    # -------------------------------------------------
    # 1. Construir modelo CBAM (idéntico al entrenamiento)
    # -------------------------------------------------
    model = build_frcnn_resnet_cbam(DEVICE)

    # -------------------------------------------------
    # 2. Cargar checkpoint
    # -------------------------------------------------
    try:
        state = torch.load(os.path.join(MODELS_DIR, fname), map_location=DEVICE)

        # Comprobación fuerte: el checkpoint DEBE contener CBAM
        cbam_keys = [k for k in state.keys() if "cbam" in k.lower()]
        if len(cbam_keys) == 0:
            print("Checkpoint SIN pesos CBAM → omitido")
            skipped.append(fname)
            continue

        missing, unexpected = model.load_state_dict(state, strict=False)

        print(f"   missing: {len(missing)} | unexpected: {len(unexpected)}")

        # Demasiadas incompatibilidades entonces evaluación inválida
        if len(missing) > 50 or len(unexpected) > 50:
            print("Arquitectura incompatible (muchos missing/unexpected)")
            skipped.append(fname)
            continue

    except Exception as e:
        print("Error cargando:", e)
        skipped.append(fname)
        continue

    # -------------------------------------------------
    # 3. Evaluación NODE21
    # -------------------------------------------------
    t0 = time.time()
    out = evaluate_model(model, val_dataset, DEVICE, fname)

    if out.get("early_stopped"):
        print("Early stopped (modelo degenerado)")
        skipped.append(fname)
        continue

    # -------------------------------------------------
    # 4. Guardar figura FROC
    # -------------------------------------------------
    fig_path = os.path.join(
        OUTPUT_DIR, fname.replace(".pth", "_froc.png")
    )
    out["figure"].savefig(fig_path, dpi=300)

    # -------------------------------------------------
    # 5. Registrar resultados
    # -------------------------------------------------
    results.append({
        "model": fname,
        "node21_score": out["node21_score"],
        "time_sec": round(time.time() - t0, 1),
        "missing": len(missing),
        "unexpected": len(unexpected),
        **{f"sens@{fp}": s for fp, s in out["sensitivities"].items()}
    })

    print(f"NODE21 score: {out['node21_score']:.4f}")

print(f"\n⏱Tiempo total: {(time.time() - start_all)/60:.1f} min")
print(f"Modelos válidos: {len(results)}")
print(f"Modelos descartados: {len(skipped)}")


### Guardat i rànquing final de models (CSV + millor checkpoint)
Aquest fragment converteix els resultats de l’avaluació en un **DataFrame**, els ordena pel **NODE21 score** de major a menor i els guarda en un fitxer **CSV** per comparar-los fàcilment.

A més:
- Mostra per pantalla quin és el **millor model** (el primer del rànquing).
- Llista tots els models **descartats** durant el procés (incompatibles o *early-stopped*).


In [64]:
df = pd.DataFrame(results).sort_values("node21_score", ascending=False)
df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison_vindn_attention_cbam.csv"), index=False)

print("\n Mejor modelo:")
print(df.iloc[0])

print("\n Modelos descartados:")
for m in skipped:
    print(" -", m)



 Mejor modelo:
model           checkpoint_canal_frcnn_vindn_attention_cbam_ep...
node21_score                                             0.892441
time_sec                                                    189.3
missing                                                         0
unexpected                                                      0
sens@0.25                                                0.762238
sens@0.5                                                 0.891109
sens@1                                                   0.916084
sens@2                                                   0.927073
sens@4                                                   0.929071
sens@8                                                   0.929071
Name: 16, dtype: object

 Modelos descartados:


### Atenció SE al cap ROI (ROIBoxHeadWithAttention)
Aquest codi afegeix un mecanisme d’atenció tipus **Squeeze-and-Excitation (SE)** al **cap de ROI** de Faster R-CNN.

Primer, `SEBlock1D` calcula uns **pesos per canal** (vector) i reescala les característiques perquè el model doni més importància als components més rellevants.

Després, `ROIBoxHeadWithAttention` aplica el `TwoMLPHead` original per obtenir un vector de **1024 features** per cada ROI i tot seguit hi aplica el SE, millorant la representació abans de la classificació i regressió de caixes.


In [65]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        w = F.relu(self.fc1(x))
        w = torch.sigmoid(self.fc2(w))
        return x * w


class ROIBoxHeadWithAttention(nn.Module):
    def __init__(self, box_head):
        super().__init__()
        self.box_head = box_head          # TwoMLPHead original
        self.attn = SEBlock1D(1024)

    def forward(self, x):
        x = self.box_head(x)              # [N, 1024]
        x = self.attn(x)
        return x


### Faster R-CNN ResNet50+FPN amb ROI Attention i pesos VinDR
Aquesta funció construeix un **Faster R-CNN (ResNet50 + FPN)** i inicialitza el model amb pesos preentrenats de **VinDR**, eliminant la capçalera original de classificació/regressió (head) perquè no coincideix amb les classes de NODE21.

Després, modifica el mòdul `roi_heads.box_head` afegint una capa d’atenció **SE (ROI Attention)** sobre el vector de característiques de cada ROI, per reforçar les parts més informatives abans de fer la predicció.

Finalment, substitueix el head per un de nou amb **2 classes** (fons + nòdul), carrega el model al dispositiu i el deixa llest per a inferència o fine-tuning.


In [71]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def build_frcnn_resnet_roi_attention(device):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None
    )

    # -----------------------------------
    # Cargar pesos VinDR (SIN HEAD)
    # -----------------------------------


    MODEL_PATH = "/content/drive1/MyDrive/ML/DATASETS/NODE21/models/fastercnn50_vindr.pth"
    state_dict = torch.load(MODEL_PATH, map_location=device)

    keys_to_remove = [
        "roi_heads.box_predictor.cls_score.weight",
        "roi_heads.box_predictor.cls_score.bias",
        "roi_heads.box_predictor.bbox_pred.weight",
        "roi_heads.box_predictor.bbox_pred.bias",
    ]
    for k in keys_to_remove:
        state_dict.pop(k, None)

    model.load_state_dict(state_dict, strict=False)

    # -----------------------------------
    # ROI ATTENTION 
    # -----------------------------------
    old_roi = model.roi_heads
    old_roi.box_head = ROIBoxHeadWithAttention(old_roi.box_head)

    # -----------------------------------
    # Head NODE21
    # -----------------------------------
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)

    model.to(device)
    model.eval()
    return model


### Avaluació automàtica de checkpoints ROI-Attention (FROC + NODE21 score)
Aquest bloc busca tots els *checkpoints* `.pth` del directori `MODELS_DIR` que continguin `"roi"` al nom, i els avalua un per un.

Per a cada model:

- Construeix de nou l’arquitectura **Faster R-CNN + ROI Attention** (igual que en entrenament).
- Carrega el checkpoint (guardant les diferències `missing/unexpected` si n’hi ha).
- Executa l’avaluació sobre `val_dataset` mitjançant **FROC**, i calcula el **NODE21 score**.
- Si el model és “degenerat” (massa dolent), activa *early stop* i el descarta.
- Desa la corba **FROC** com a imatge i guarda les mètriques principals dins `results`.

Al final, `results` conté el rànquing de models vàlids i `skipped` els descartats.


In [ ]:
results = []
skipped = []


model_files = [
    f for f in os.listdir(MODELS_DIR)
    if f.endswith(".pth") and "roi" in f.lower()
]

start_all = time.time()

for fname in tqdm(model_files, desc="Evaluando modelos ROI-attention"):
    print(f"\n {fname}")

    # -------------------------------------------------
    # 1. Construir modelo ROI-attention
    # -------------------------------------------------
    model = build_frcnn_resnet_roi_attention(DEVICE)

    # -------------------------------------------------
    # 2. Cargar checkpoint
    # -------------------------------------------------
    try:
        state = torch.load(os.path.join(MODELS_DIR, fname), map_location=DEVICE)
        missing, unexpected = model.load_state_dict(state, strict=False)

        if missing or unexpected:
            print(f"   missing keys: {len(missing)}")
            print(f"   unexpected keys: {len(unexpected)}")

    except Exception as e:
        print(" Error cargando:", e)
        skipped.append(fname)
        continue

    # -------------------------------------------------
    # 3. Evaluación NODE21
    # -------------------------------------------------
    t0 = time.time()
    out = evaluate_model(model, val_dataset, DEVICE, fname)

    if out.get("early_stopped"):
        print(" Early stopped")
        skipped.append(fname)
        continue

    # -------------------------------------------------
    # 4. Guardar resultados
    # -------------------------------------------------
    out["figure"].savefig(
        os.path.join(OUTPUT_DIR, fname.replace(".pth", "_froc.png")),
        dpi=300
    )

    results.append({
        "model": fname,
        "node21_score": out["node21_score"],
        "time_sec": round(time.time() - t0, 1),
        **{f"sens@{fp}": s for fp, s in out["sensitivities"].items()}
    })

    print(f" NODE21 score: {out['node21_score']:.4f}")


### Avaluació automàtica de checkpoints CBAM (FROC + NODE21 score)
Aquest bloc recorre tots els fitxers `.pth` de `MODELS_DIR` que continguin `"cbam"` al nom i els avalua sobre `val_dataset`.

Per cada checkpoint:

- Construeix el model **Faster R-CNN + CBAM** (mateixa arquitectura que l’entrenament).
- Carrega els pesos del checkpoint (si falla, es descarta).
- Avalua el model amb `evaluate_model()` per obtenir la corba **FROC** i el **NODE21 score**.
- Si el model és massa dolent, activa *early stop* i es guarda dins `skipped`.
- Desa la figura FROC en PNG i registra resultats (temps + sensibilitats) dins `results`.

Al final, `results` conté els models vàlids amb les seves mètriques i `skipped` els descartats.


In [ ]:
results = []
skipped = []

model_files = [
    f for f in os.listdir(MODELS_DIR)
    if f.endswith(".pth") and "cbam" in f.lower()
]

start_all = time.time()

for fname in tqdm(model_files, desc="Evaluando modelos CBAM"):
    print(f"\n {fname}")

    model = build_frcnn_resnet_cbam(DEVICE)

    try:
        state = torch.load(os.path.join(MODELS_DIR, fname), map_location=DEVICE)
        model.load_state_dict(state, strict=False)
    except Exception as e:
        print(" Error cargando:", e)
        skipped.append(fname)
        continue

    t0 = time.time()
    out = evaluate_model(model, val_dataset, DEVICE, fname)

    if out.get("early_stopped"):
        print(" Early stopped")
        skipped.append(fname)
        continue

    out["figure"].savefig(
        os.path.join(OUTPUT_DIR, fname.replace(".pth", "_froc.png")),
        dpi=300
    )

    results.append({
        "model": fname,
        "node21_score": out["node21_score"],
        "time_sec": round(time.time() - t0, 1),
        **{f"sens@{fp}": s for fp, s in out["sensitivities"].items()}
    })

    print(f" NODE21 score: {out['node21_score']:.4f}")


### Avaluació de checkpoints “Local Focal Loss” amb FROC i NODE21 score
Aquest codi avalua automàticament tots els models `.pth` de `MODELS_DIR` que contenen `"local"` al nom.

Per cada checkpoint:

- Construeix el model base compatible amb *Local Focal Loss* (`build_model_for_local_focal`).
- Carrega els pesos del checkpoint amb `strict=False` (permet petites diferències).
- Avalua el model amb `evaluate_model()` sobre `val_dataset` (calcula FROC i NODE21 score).
- Si el model és dolent (*early stop*), es descarta.
- Desa la figura FROC en PNG i guarda les mètriques a `results`.

Finalment mostra el temps total d’avaluació.


In [ ]:
results = []
skipped = []

model_files = [
    f for f in os.listdir(MODELS_DIR)
    if f.endswith(".pth") and "local" in f.lower()
]

start_all = time.time()

for fname in tqdm(model_files, desc="Evaluando modelos Local Focal Loss"):
    print(f"\n {fname}")

    model = build_model_for_local_focal(DEVICE)

    # cargar checkpoint (strict=False es OK)
    try:
        state = torch.load(os.path.join(MODELS_DIR, fname), map_location=DEVICE)
        model.load_state_dict(state, strict=False)
    except Exception as e:
        print(" Error cargando:", e)
        skipped.append(fname)
        continue

    t0 = time.time()
    out = evaluate_model(model, val_dataset, DEVICE, fname)

    if out.get("early_stopped"):
        print(" Early stopped")
        skipped.append(fname)
        continue

    out["figure"].savefig(
        os.path.join(OUTPUT_DIR, fname.replace(".pth", "_froc.png")),
        dpi=300
    )

    results.append({
        "model": fname,
        "node21_score": out["node21_score"],
        "time_sec": round(time.time() - t0, 1),
        **{f"sens@{fp}": s for fp, s in out["sensitivities"].items()}
    })

    print(f" NODE21 score: {out['node21_score']:.4f}")

print(f"\n⏱ Total tiempo: {(time.time()-start_all)/60:.1f} min")



### Avaluació de checkpoints “Local Focal Loss” amb FROC i NODE21 score
Aquest codi avalua automàticament tots els models `.pth` de `MODELS_DIR` que contenen `"local"` al nom.

Per cada checkpoint:

- Construeix el model base compatible amb *Local Focal Loss* (`build_model_for_local_focal`).
- Carrega els pesos del checkpoint amb `strict=False` (permet petites diferències).
- Avalua el model amb `evaluate_model()` sobre `val_dataset` (calcula FROC i NODE21 score).
- Si el model és dolent (*early stop*), es descarta.
- Desa la figura FROC en PNG i guarda les mètriques a `results`.

Finalment mostra el temps total d’avaluació.


### Comparació i exportació dels resultats dels models (ranking final)
Aquest bloc converteix `results` en un **DataFrame**, l’ordena pel **NODE21 score** (de millor a pitjor) i exporta la comparativa a un fitxer CSV.

Després:

- Mostra per pantalla el **millor model** (`df.iloc[0]`).
- Llista els **models descartats** durant l’avaluació (`skipped`), normalment per errors de càrrega o *early stop*.


In [ ]:
df = pd.DataFrame(results).sort_values("node21_score", ascending=False)
df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison_vindn_roi_.csv"), index=False)

print("\n Mejor modelo:")
print(df.iloc[0])

print("\n Modelos descartados:")
for m in skipped:
    print(" -", m)


In [ ]:
os.listdir("/content/drive1/MyDrive/ML/DATASETS/NODE21")

['proccessed_data',
 'augmented_pos',
 'metadata_balanced.csv',
 'efficientnet_b0_multicanal.pth',
 'efficientnet_b0_multicanal_laplaciana.pth',
 'efficientnet_b0_multicanal_Unsharp.pth',
 'efficientnet_b0_multicanal_Unsharp_bo.pth',
 'original_data',
 'original_augmented_pos',
 'metadata_original_balanced.csv',
 'best_frcnn_node21_1024.pth',
 'best_frcnn_node21.pth',
 'metadata_balanced.gsheet',
 'models',
 'MODELS1',
 'RESULTS']